# Step 2 on Kaggle: train FaST-ISL (resumes across sessions)

**Settings (right-hand panel) before running:**
- *Session options → Accelerator:* **GPU T4 x2** (one GPU is used)
- *Session options → Internet:* **On**
- *Input → Add Input → Your Work → Notebooks:* add the **output of the step-1 conversion notebook**
- **From the 2nd run on:** also add the **output of this notebook's previous version**
  (*Add Input → Your Work → Notebooks → this notebook*). That is how training resumes: the runs folder of
  the last session is copied back in. If Kaggle does not offer it, open the previous version's *Output* tab
  → **New Dataset**, and add that dataset instead.

Click **Save Version → Save & Run All (Commit)**. It runs in the background for up to ~11 hours, then stops
cleanly. Each new commit continues where the last one stopped: finished models are skipped and the
current one resumes from its last epoch. About 30 GPU hours per week are free.

Both GPUs are used: the baseline (`B0_body_hands`) and the full model (`FaST`) train side by side, then
the ablations, then the sentence-boundary detector and the continuous-signing evaluation.
Results go to `runs_v2/` (v2 settings: word dropout + contrastive loss; see `configs/isign_base.yaml`).

In [ ]:
import time; NOTEBOOK_START = time.time()
BUDGET_H = 11.0   # Kaggle stops GPU sessions at 12 h; leave room to save the output
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /tmp/ISL && git clone -q https://github.com/sengarshivansh/ISL.git /tmp/ISL
!pip -q install sacrebleu rouge-score sentencepiece pyyaml
%cd /tmp/ISL

In [ ]:
import glob, os, shutil
# keypoint shards from the step-1 notebook
hits = glob.glob('/kaggle/input/**/shard_00000.npz', recursive=True)
assert hits, 'Add the output of the step-1 conversion notebook as an input (see the instructions above)'
DATA = os.path.dirname(hits[0])
print(len(glob.glob(f'{DATA}/shard_*.npz')), 'shards in', DATA)

# point every step at the Kaggle paths
cfg_path = 'configs/isign_base.yaml'
text = open(cfg_path).read().replace('/content/drive/MyDrive/ISL/isign_pose_keypoints', DATA)
open(cfg_path, 'w').write(text)

# runs from earlier sessions (output of a previous version of this notebook)
RUNS = '/kaggle/working/runs_v2'   # v2 = word dropout + contrastive loss (v1 runs are ignored)
prev = [p for p in glob.glob('/kaggle/input/**/runs_v2', recursive=True) if os.path.isdir(p)]
if prev and not os.path.exists(RUNS):
    shutil.copytree(prev[0], RUNS)
    print('resuming from', prev[0])
os.makedirs(RUNS, exist_ok=True)
for d in sorted(os.listdir(RUNS)):
    state = 'done' if os.path.exists(f'{RUNS}/{d}/results.json') else 'in progress'
    print(f'  {d}: {state}')

In [ ]:
import subprocess, time, sys
def hours_left():
    return BUDGET_H - (time.time() - NOTEBOOK_START) / 3600

# Two GPUs, two queues: the baseline and the full model train side by side, then the ablations.
QUEUES = [['B0_body_hands', 'B1_speedaug', 'FaST_concat', 'F_mouth', 'F_eyes'],
          ['FaST', 'FaST_noaug', 'F_eyebrows', 'F_head']]
import torch
n_gpu = torch.cuda.device_count()
if n_gpu < 2:                        # one GPU: a single queue in the same priority order
    QUEUES = [[n for pair in zip(QUEUES[0], QUEUES[1] + [None]) for n in pair if n]]
print(n_gpu, 'GPU(s);', QUEUES)

WORKER = r"""
import os, sys, time, yaml
from islt.train import load_config, deep_update, train
runs, deadline, names = sys.argv[1], float(sys.argv[2]), sys.argv[3:]
for name in names:
    if os.path.exists(f"{runs}/{name}/results.json"):
        print("done:", name, flush=True); continue
    left = (deadline - time.time()) / 3600
    if left < 1.0:
        print("not enough time left: commit again to continue", flush=True); break
    print(f"=== training {name} ({left:.1f} h left)", flush=True)
    cfg = load_config("configs/isign_base.yaml")
    deep_update(cfg, yaml.safe_load(open(f"configs/ablations/{name}.yaml")))
    cfg["out_dir"] = runs
    cfg["train"]["max_hours"] = round(left, 2)
    train(cfg)
    if not os.path.exists(f"{runs}/{name}/results.json"):
        print(f"{name} paused at the time limit: commit again to continue", flush=True); break
"""
open('/tmp/worker.py', 'w').write(WORKER)
deadline = NOTEBOOK_START + BUDGET_H * 3600
procs = []
for gpu, names in enumerate(QUEUES):
    log = open(f'{RUNS}/worker{gpu}.log', 'a')
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'PYTHONPATH': '/tmp/ISL'}
    procs.append((subprocess.Popen([sys.executable, '/tmp/worker.py', RUNS, str(deadline), *names],
                                   stdout=log, stderr=subprocess.STDOUT, env=env, cwd='/tmp/ISL'), gpu))

# print each worker's newest epoch line every 10 minutes, so progress shows in the Kaggle log
while any(p.poll() is None for p, _ in procs):
    time.sleep(600)
    for p, gpu in procs:
        lines = [l for l in open(f'{RUNS}/worker{gpu}.log') if l.strip()]
        print(f'[GPU {gpu}] {lines[-1].strip()[:300] if lines else "starting"}', flush=True)
for p, gpu in procs:
    print(f'--- GPU {gpu} finished (exit code {p.returncode}); last lines:')
    print(''.join(open(f'{RUNS}/worker{gpu}.log').readlines()[-15:]))

In [ ]:
# sentence-boundary detector (about 20 min) and continuous-signing evaluation, once the main models exist
main_done = all(os.path.exists(f'{RUNS}/{n}/results.json') for n in ['B0_body_hands', 'FaST'])
if main_done and not os.path.exists(f'{RUNS}/boundary/boundary.pt') and hours_left() > 0.7:
    !python -m islt.boundary --config configs/isign_base.yaml --out {RUNS}/boundary --steps 6000 --device cuda
if main_done and os.path.exists(f'{RUNS}/boundary/boundary.pt'):
    for run in ['FaST', 'B0_body_hands']:
        if hours_left() < 0.5: break
        if not os.path.exists(f'{RUNS}/{run}/stream_results.json'):            # pause-based segmenter
            !python -m islt.eval_stream --config configs/isign_base.yaml --checkpoint {RUNS}/{run}/best.pt
        if not os.path.exists(f'{RUNS}/{run}/stream_results_learned.json'):    # learned boundary detector
            !python -m islt.eval_stream --config configs/isign_base.yaml --checkpoint {RUNS}/{run}/best.pt --boundary {RUNS}/boundary/boundary.pt

In [ ]:
!python -m islt.report {RUNS} --out /kaggle/working/results_table.tex
!ls -la {RUNS}/*/